<a href="https://colab.research.google.com/github/mvharsh/Mathematical-Computing/blob/main/Two_Phase_Simplex_LPP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SOLVING LINEAR PROGRAMMING PROBLEMS USING TWO-PHASE SIMPLEX METHOD**

**AIM:**

To solve a Linear Programming Problem (LPP) using the Two-Phase Simplex Method in Python, handling both minimization and maximization problems by converting constraints into standard form and utilizing scipy.optimize.linprog.


**ALGORITHM:**

**Phase 1: Check Feasibility**

1.	Convert inequality constraints into equalities by adding slack, surplus, and artificial variables.
2.	Formulate an auxiliary objective function by minimizing the sum of artificial variables.
3.	Apply the Simplex Method to find a basic feasible solution.
4.	If the minimum value of the auxiliary objective function is zero, proceed to Phase 2; otherwise, the problem is infeasible.

**Phase 2: Solve the Original Problem**

5.	Remove artificial variables and restore the original objective function.
6.	Use the Simplex Method again to find the optimal solution.
7.	Output the optimal value and decision variables.


In [ ]:
import numpy as np
from scipy.optimize import linprog

def two_phase_simplex(c, A_eq, b_eq, A_ub=None, b_ub=None):
    num_vars = len(c)

    # Ensure A_eq and b_eq are not None
    A_eq = np.array(A_eq) if A_eq else np.empty((0, num_vars))
    b_eq = np.array(b_eq) if b_eq else np.empty(0)

    # Phase 1 only needed if there are equality constraints
    if len(b_eq) > 0:
        num_eq = len(b_eq)
        c_phase1 = np.concatenate([np.zeros(num_vars), np.ones(num_eq)])
        A_phase1 = np.hstack([A_eq, np.eye(num_eq)])
        b_phase1 = b_eq

        # Solve phase 1
        result_phase1 = linprog(c_phase1, A_eq=A_phase1, b_eq=b_phase1, method='simplex')

        if not result_phase1.success or result_phase1.fun > 1e-8:
            print("Phase 1: No feasible solution")
            return result_phase1

        print("Phase 1 complete: Feasible solution found")

    # Ensure A_ub and b_ub are not None
    A_ub = np.array(A_ub) if A_ub else np.empty((0, num_vars))
    b_ub = np.array(b_ub) if b_ub else np.empty(0)

    # Phase 2: Solve original problem with feasible starting point
    result_phase2 = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, method='highs')

    if result_phase2.success:
        print("Phase 2 complete: Optimal solution found")
        print(f"Objective value: {result_phase2.fun}")
        print(f"Variable values: {result_phase2.x}")
    else:
        print("Phase 2 failed:", result_phase2.message)

    return result_phase2

if __name__ == "__main__":
    # Get user input
    num_vars = int(input("Enter the number of variables: "))
    optimization_type = input("Do you want to maximize or minimize the objective function? (enter 'max' or 'min'): ").strip().lower()
    num_eq = int(input("Enter the number of equality constraints: "))
    num_ineq = int(input("Enter the number of inequality constraints: "))

    c = list(map(float, input(f"Enter the coefficients of the objective function (space-separated, {num_vars} values): ").split()))

    # Flip sign of objective function for maximization
    if optimization_type == 'max':
        c = [-x for x in c]

    A_eq = []
    b_eq = []
    if num_eq > 0:
        for i in range(num_eq):
            A_eq.append(list(map(float, input(f"Enter equality constraint {i+1} coefficients (space-separated, {num_vars} values): ").split())))
        b_eq = list(map(float, input(f"Enter the RHS of equality constraints (space-separated, {num_eq} values): ").split()))

    A_ub = []
    b_ub = []
    if num_ineq > 0:
        for i in range(num_ineq):
            sense = input(f"Is inequality constraint {i+1} <= or >= (enter '<=' or '>='): ")
            coeffs = list(map(float, input(f"Enter inequality constraint {i+1} coefficients (space-separated, {num_vars} values): ").split()))
            rhs = float(input(f"Enter the RHS of inequality constraint {i+1}: "))

            if sense == '>=':
                coeffs = [-x for x in coeffs]
                rhs = -rhs

            A_ub.append(coeffs)
            b_ub.append(rhs)

    result = two_phase_simplex(c, A_eq, b_eq, A_ub, b_ub)

    # Flip the objective function value back if it was maximization
    if optimization_type == 'max' and result.success:
        print(f"Maximized objective value: {-result.fun}")


Enter the number of variables: 2
Do you want to maximize or minimize the objective function? (enter 'max' or 'min'): max
Enter the number of equality constraints: 0
Enter the number of inequality constraints: 2
Enter the coefficients of the objective function (space-separated, 2 values): 4 3
Is inequality constraint 1 <= or >= (enter '<=' or '>='): <=
Enter inequality constraint 1 coefficients (space-separated, 2 values): 1 1
Enter the RHS of inequality constraint 1: 5
Is inequality constraint 2 <= or >= (enter '<=' or '>='): >=
Enter inequality constraint 2 coefficients (space-separated, 2 values): 2 3
Enter the RHS of inequality constraint 2: 6
Phase 2 complete: Optimal solution found
Objective value: -20.0
Variable values: [5. 0.]
Maximized objective value: 20.0


## **Another code**

**•  Input:**

  •	Coefficients of the objective function c

  •	Coefficients of the constraints A

  •	Right-hand side values b

**•  Phase 1:**

  a. Add artificial variables to convert inequalities to equalities.

  b. Construct the initial tableau with artificial variables and an auxiliary objective function.

  c. Use the simplex method to minimize the auxiliary objective.

  d. If the minimum value of the auxiliary objective is not zero, the problem is infeasible — stop here.

**•  Phase 2:**

  a. Remove artificial variables and construct the original objective function tableau.

  b. Apply the simplex method to find the optimal solution.

**•  Output:**

  •	Final tableau

  •	Optimal solution for the decision variables

  •	Maximum value of the objective function Z

  •	Indicate whether the solution is optimal or not


In [ ]:
import numpy as np

def simplex(c, A, b, phase=1):
    m, n = A.shape
    tableau = np.hstack([A, np.eye(m), b.reshape(-1, 1)])

    if phase == 1:
        c = np.hstack([np.zeros(n), np.ones(m), [0]])
    else:
        c = np.hstack([c, np.zeros(m + 1)])

    tableau = np.vstack([tableau, c])

    while np.any(tableau[-1, :-1] < 0):
        pivot_col = np.argmin(tableau[-1, :-1])
        ratios = np.divide(tableau[:-1, -1], tableau[:-1, pivot_col],
                           out=np.full_like(tableau[:-1, -1], np.inf),
                           where=tableau[:-1, pivot_col] > 0)
        pivot_row = np.argmin(ratios)

        if ratios[pivot_row] == np.inf:
            return "Unbounded"

        tableau[pivot_row] /= tableau[pivot_row, pivot_col]

        for i in range(len(tableau)):
            if i != pivot_row:
                tableau[i] -= tableau[i, pivot_col] * tableau[pivot_row]

    return tableau

def two_phase_simplex(c, A, b):
    tableau = simplex(c, A, b, phase=1)

    if not np.isclose(tableau[-1, -1], 0):
        return "Infeasible"

    m = len(b)
    tableau = simplex(c, tableau[:-1, :-1 - m], tableau[:-1, -1], phase=2)

    solution = {f'x{i + 1}': 0 for i in range(len(c))}

    for i in range(len(A)):
        basic_var = np.where(tableau[i, :-1] == 1)[0]
        if len(basic_var) == 1 and basic_var[0] < len(c):
            solution[f'x{basic_var[0] + 1}'] = tableau[i, -1]

    max_z = tableau[-1, -1]
    is_optimal = not np.any(tableau[-1, :-1] < 0)

    return tableau, solution, max_z, is_optimal

num_vars = int(input("Enter the number of variables: "))
num_constraints = int(input("Enter the number of constraints: "))

print("Enter the coefficients of the objective function (separated by space):")
c = np.array(list(map(float, input().split()))) * -1  # Convert max to min

A = []
b = []

print("Enter the constraints in the format: coefficients followed by inequality (<=, >=, =) and RHS value")
for _ in range(num_constraints):
    *coefficients, inequality, rhs = input().split()
    coefficients = list(map(float, coefficients))
    rhs = float(rhs)

    if inequality == '>=':
        coefficients = [-x for x in coefficients]
        rhs = -rhs

    A.append(coefficients)
    b.append(rhs)

A = np.array(A)
b = np.array(b)

result_matrix, solution, max_z, is_optimal = two_phase_simplex(c, A, b)

print("\nResulting Tableau:")
print(result_matrix)
print("Optimal Solution:", solution)
print("Max Z Value:", max_z)
print("Is the current solution optimal?", is_optimal)


Enter the number of variables: 3
Enter the number of constraints: 3
Enter the coefficients of the objective function (separated by space):
2 1 0.25
Enter the constraints in the format: coefficients followed by inequality (<=, >=, =) and RHS value
4 6 3 <= 8
3 -6 -4 <= 1
2 3 -5 >= 4

Resulting Tableau:
[[ 0.          1.          0.5952381   0.07142857 -0.0952381   0.
   0.47619048]
 [ 1.          0.         -0.14285714  0.14285714  0.14285714  0.
   1.28571429]
 [ 0.          0.          6.5         0.5         0.          1.
   0.        ]
 [ 0.          0.          0.05952381  0.35714286  0.19047619  0.
   3.04761905]]
Optimal Solution: {'x1': 1.2857142857142858, 'x2': 0.4761904761904762, 'x3': 0}
Max Z Value: 3.0476190476190474
Is the current solution optimal? True
